In [1]:
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
load_dotenv()

model = init_chat_model("google_genai:gemini-3.6-flash")

In [2]:

from langchain_mistralai import ChatMistralAI

mis_model = ChatMistralAI(
    model="mistral-medium-3-5",
    temperature=0,
    model_kwargs={"reasoning_effort": "high"},
)

In [3]:
import time
from langchain.tools import tool
from langgraph.config import get_stream_writer
from deepagents import create_deep_agent, CompiledSubAgent
from langgraph.graph import StateGraph, START, END, MessagesState


# --- Tool used by the nested deep agent, emits its own progress ---
@tool
def analyze_data(topic: str) -> str:
    """Run a data analysis on a given topic."""
    writer = get_stream_writer()
    writer({"status": "started", "topic": topic})
    time.sleep(0.3)
    writer({"status": "halfway", "progress": 50})
    time.sleep(0.3)
    writer({"status": "done", "progress": 100})
    return f'Analysis of "{topic}": sentiment is 85% positive.'


# --- The deep agent living INSIDE node2 ---
nested_agent = create_deep_agent(
    model=model,
    system_prompt="You are a data analyst. Call analyze_data for every request.",
    tools=[analyze_data],
)


# --- Pipeline (the "compiled subagent" of main_agent) ---
class PipelineState(MessagesState):
    intermediate: str | None


def node1(state: PipelineState):
    return {"intermediate": "step 1 done"}


def node2(state: PipelineState):
    # plain .invoke() — LangGraph auto-nests this, streaming still sees inside it
    result = nested_agent.invoke(
        {"messages": [{"role": "user", "content": "Analyze customer satisfaction trends"}]}
    )
    return {"intermediate": result["messages"][-1].content}


def node3(state: PipelineState):
    return {"messages": [{"role": "assistant", "content": state["intermediate"]}]}


def build_pipeline():
    g = StateGraph(PipelineState)
    g.add_node("node1", node1)
    g.add_node("node2", node2)
    g.add_node("node3", node3)
    g.add_edge(START, "node1")
    g.add_edge("node1", "node2")
    g.add_edge("node2", "node3")
    g.add_edge("node3", END)
    return g.compile()


pipeline_subagent = CompiledSubAgent(
    name="pipeline",
    description="Runs a pipeline; step 2 delegates to a specialist analyst agent.",
    runnable=build_pipeline(),
)

main_agent = create_deep_agent(
    model=mis_model,
    system_prompt="Call the pipeline subagent for any analysis task.",
    subagents=[pipeline_subagent],
)

In [5]:
pending_by_parent = {}   # parent_ns -> [subagent_type, ...] dispatched but not yet labeled
resolved = {}             # ns prefix -> resolved label
current_source = None     # tracks source changes for clean token grouping


def strip_id(segment: str) -> str:
    return segment.split(":", 1)[0]


def track_dispatches(ns, node_name, node_data):
    if node_name != "model":
        return
    for msg in node_data.get("messages", []):
        for tc in getattr(msg, "tool_calls", []):
            if tc["name"] == "task":
                pending_by_parent.setdefault(ns, []).append(tc["args"].get("subagent_type", "subagent"))


def label_path(ns) -> str:
    if not ns:
        return "main agent"
    path = []
    for i, segment in enumerate(ns):
        prefix, key = ns[:i], ns[:i + 1]
        if segment.startswith("tools:"):
            if key not in resolved:
                queue = pending_by_parent.get(prefix, [])
                resolved[key] = queue.pop(0) if queue else "subagent"
            path.append(resolved[key])
        else:
            path.append(strip_id(segment))
    return " > ".join(path)


for chunk in main_agent.stream(
    {"messages": [{"role": "user", "content": "Run the pipeline analysis."}]},
    stream_mode=["updates", "messages", "custom"],
    subgraphs=True,
    version="v2",
):
    ns = chunk["ns"]
    source = label_path(ns)

    if chunk["type"] == "updates":
        for node_name, node_data in chunk["data"].items():
            track_dispatches(ns, node_name, node_data)
            print(f"[{source}] step: {node_name}")
            print(f"[{source}] data: {node_data}")

    elif chunk["type"] == "custom":
        print(f"[{source}] custom: {chunk['data']}")

    elif chunk["type"] == "messages":
        token, metadata = chunk["data"]

        if source != current_source:
            print(f"\n--- [{source}] ---")
            current_source = source

        for block in getattr(token, "content_blocks", []):
            if block["type"] == "reasoning" and block.get("reasoning"):
                print(f"\033[2m{block['reasoning']}\033[0m", end="", flush=True)  # dim = thinking
            elif block["type"] == "text" and block.get("text"):
                print(block["text"], end="", flush=True)

print()

[main agent] step: PatchToolCallsMiddleware.before_agent
[main agent] data: None

--- [main agent] ---
The user wants me to run a pipeline analysis. According to the instructions, I should call the pipeline subagent for any analysis task. Let me use the task tool with subagent_type="pipeline" to run the pipeline analysis.[main agent] step: model
[main agent] data: {'messages': [AIMessage(content=['', {'type': 'thinking', 'thinking': [{'type': 'text', 'text': 'The user wants me to run a pipeline analysis. According to the instructions, I should call the pipeline subagent for any analysis task. Let me use the task tool with subagent_type="pipeline" to run the pipeline analysis.', 'index': 0}], 'closed': True, 'index': 0}], additional_kwargs={'tool_calls': [{'id': 'l8pl4nTQO', 'type': 'function', 'function': {'name': 'task', 'arguments': '{"description": "Run the pipeline analysis task", "subagent_type": "pipeline"}'}, 'index': 0}]}, response_metadata={'model_provider': 'mistralai', 'mode